# audiblez on Kaggle — audiobook rendering on 2× T4 (free)

**Before running** (once): *Settings → Accelerator → GPU T4 x2*, *Settings → Internet → On* (needs a phone-verified account),
then *Add Input* → the dataset that contains your story (`.md`, `.txt`, `.epub`, or a folder with one file per chapter).

**Chạy nền (khuyên dùng):** *Save Version → Save & Run All (Commit)* — notebook chạy tiếp cả khi đóng trình duyệt;
kết quả nằm trong tab *Output* (`output/…mp4`, `…m4b`, `…_youtube.txt`). Một phiên tối đa 12 giờ; nếu chưa xong,
gắn output của lần chạy trước làm Input và đặt `RESUME_FROM` bên dưới để chạy tiếp từ cache.


In [ ]:
# ===== Settings / Cấu hình =====
BOOK = '/kaggle/input/my-story/story.md'   # .md / .txt / .docx / .fb2 / .epub file, or a folder of .txt/.md
                                           # files (one per chapter); a list of books renders them one after
                                           # the other within the session, e.g. a week of daily stories
VOICE = 'ja_male_narrator'                 # ja_male_narrator, ja_female_narrator, ko_male_narrator, ko_female_narrator,
                                           # ru_male_narrator, ... or a voice-profile folder from your dataset
EPISODE_HOURS = 2                          # split into ~2 h YouTube videos (0 = one file)
EXTRA = ''                                 # e.g. '--lexicon /kaggle/input/my-story/lexicon.txt --chapters 1-5'
STRESS = False                             # Russian: automatic word stress (RUAccent + a Qwen3-TTS fine-tune
                                           # that follows it); listen to one chapter with and without first
RUN_BENCHMARK = True                       # measure the fastest precision/backend/batch first (~10-15 min)
BACKEND, BATCH, DTYPE = 'auto', 0, 'auto'  # used when RUN_BENCHMARK = False: paste the 'Fastest' values of a
                                           # previous benchmark, e.g. 'faster', 1, 'bfloat16'
REVIEW = True                              # second opinion of Whisper large-v3-turbo on every sentence; the
                                           # doubtful ones get 4 alternative takes, the best is kept (review.txt)
CHAPTER_CARDS = True                       # video: the chapter title over the picture while it is read
AMBIENCE = ''                              # e.g. '/kaggle/input/my-story/sea.flac': looped quietly under the
                                           # narration (use a recording you may publish: CC0 or your own)
RESUME_FROM = ''                           # e.g. '/kaggle/input/audiblez-render/output' to resume a previous run
AUDIBLEZ_REF = 'main'                      # git branch / tag / commit of audiblez to install
OUTPUT = '/kaggle/working/output'
SESSION_HOURS, RESERVE_HOURS = 12.0, 0.6   # Kaggle stops a session after 12 h: synthesis stops cleanly before,
                                           # leaving time for mastering/encoding and saving the output
import time
T0 = time.time()
BOOKS = list(BOOK) if isinstance(BOOK, (list, tuple)) else [BOOK]


In [ ]:
# Install audiblez with the CUDA-graph backend (faster-qwen3-tts), Whisper QA, and the extras of the language
import shutil, subprocess, sys
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Keep Kaggle's preinstalled PyTorch: it is built for this machine's NVIDIA driver (a newer wheel may not see the GPU)
from importlib import metadata
with open('/tmp/torch-constraints.txt', 'w') as f:
    for pkg in ('torch', 'torchaudio', 'torchvision'):
        try:
            f.write(f'{pkg}=={metadata.version(pkg)}\n')
        except metadata.PackageNotFoundError:
            pass
EXTRAS = 'fast,asr' + (',ja' if 'ja_' in VOICE else '') + (',stress' if STRESS else '')
!pip install -q -c /tmp/torch-constraints.txt "audiblez[{EXTRAS}] @ git+https://github.com/vansyson1308/audiblez.git@{AUDIBLEZ_REF}"
if shutil.which('ffmpeg') is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), 'No GPU: select Settings → Accelerator → GPU T4 x2'


In [ ]:
# Resume: reuse the synthesis cache of a previous run (only the missing sentences are generated)
import os, glob
os.makedirs(OUTPUT, exist_ok=True)
if RESUME_FROM:
    for src in glob.glob(os.path.join(RESUME_FROM, '*.audiblez')):
        dst = os.path.join(OUTPUT, os.path.basename(src))
        print('Resuming from', src)
        shutil.copytree(src, dst, dirs_exist_ok=True)


In [ ]:
# 1. Check the text that will be read (chapters, normalized script) — nothing is synthesized
STRESS_ARG = '--stress ruaccent' if STRESS else ''
for BOOK in BOOKS:
    !audiblez "{BOOK}" -o "{OUTPUT}" --voice-profile "{VOICE}" --dry-run {STRESS_ARG} {EXTRA}


In [ ]:
# 2. Measure the fastest settings on these GPUs (precision, backend, batch size), keeping only the ones whose
#    takes pass the quality checks (~10-15 min; next time set RUN_BENCHMARK = False and
#    BACKEND, BATCH, DTYPE to the printed values)
import re
if RUN_BENCHMARK:
    cmd = (f'audiblez "{BOOKS[0]}" -o "{OUTPUT}" --voice-profile "{VOICE}" --benchmark --device cuda:0 '
           f'{STRESS_ARG} {EXTRA}')
    lines = []
    with subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:  # live output
            print(line, end='')
            lines.append(line)
    m = re.search(r'Fastest: --backend (\S+) --batch-size (\d+) --dtype (\S+)', ''.join(lines))
    if m:
        BACKEND, BATCH, DTYPE = m.group(1), int(m.group(2)), m.group(3)
print('Using --backend', BACKEND, '--batch-size', BATCH or 'auto', '--dtype', DTYPE)


In [ ]:
# 3. Render: one worker per GPU, Whisper check of every sentence (+ the review), mastering, YouTube videos
#    with chapters and subtitles (.srt: upload it in YouTube Studio → Subtitles).
#    Stops before the end of the session (--time-limit): run again (RESUME_FROM) to finish.
episodes = f'--episode-hours {EPISODE_HOURS}' if EPISODE_HOURS else ''
VIDEO = ' '.join(x for x in ['--review' if REVIEW else '', '--chapter-cards' if CHAPTER_CARDS else '',
                             f'--ambience "{AMBIENCE}"' if AMBIENCE else ''] if x)
for BOOK in BOOKS:
    left = SESSION_HOURS - RESERVE_HOURS - (time.time() - T0) / 3600
    if left < 0.2:
        print(f'Not enough session time left for {BOOK}: run the notebook again to continue.')
        continue
    LEFT = f'{left:.2f}'
    !audiblez "{BOOK}" -o "{OUTPUT}" --voice-profile "{VOICE}" --youtube {episodes} --gpus auto --backend {BACKEND} --batch-size {BATCH} --dtype {DTYPE} --time-limit {LEFT} {VIDEO} {STRESS_ARG} {EXTRA}


In [ ]:
# 4. Results (download them from the Output tab)
for path in sorted(glob.glob(os.path.join(OUTPUT, '*'))):
    if os.path.isfile(path):
        print(f'{os.path.getsize(path) / 1e6:8.1f} MB  {os.path.basename(path)}')
for txt in sorted(glob.glob(os.path.join(OUTPUT, '*_youtube.txt'))):
    print('\n=====', os.path.basename(txt)); print(open(txt, encoding='utf-8').read())
for qa in glob.glob(os.path.join(OUTPUT, '*.audiblez', 'qa_report.txt')):
    print('\n===== QA report'); print(open(qa, encoding='utf-8').read()[:3000])

for rv in glob.glob(os.path.join(OUTPUT, '*.audiblez', 'review.txt')):
    print('\n===== Review (large-v3-turbo)'); print(open(rv, encoding='utf-8').read()[:4000])